> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 6 · Notebook 06 — Trees, Monte Carlo and Greeks from simulation

**Sessions:** S6 (Numerical pricing methods) · [Lesson plan](../../docs/lessons/PART_06_FUTURES_OPTIONS_ENGINEERING.md) · graded labs in [`labs/part06/`](../../labs/part06/)

**You will:**
1. Write the backward step of a binomial tree, and price an American put.
2. See the tree converge to BSM for a European option.
3. Cut Monte Carlo error with antithetic variates.
4. Get a usable Monte Carlo delta with common random numbers.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters, so every estimate can be compared with the truth. Units: T in years, σ as a decimal, vega per 1.00 σ, theta per year, `cp = +1` call / `−1` put.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p6lib.py is in notebooks/part06/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p6lib as p

p.use_course_style()

## 1. The CRR binomial tree

Split `T` into `steps`; each step the price goes up by `u = e^{σ√dt}` or down by `d = 1/u`, with risk-neutral probability `p = (e^{(r−q)dt} − d)/(u − d)`. Start from the payoffs at expiry and step back: each node is the discounted expected value of its two children, and for an **American** option the larger of that and exercising now. At step `i` the node prices are `S·u^j·d^(i−j)` for `j = 0 … i`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def crr_price(S, K, T, r, q, sigma, cp, steps=500, american=True):
    dt = T / steps
    u = np.exp(sigma * np.sqrt(dt)); d = 1 / u
    pu = (np.exp((r - q) * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    j = np.arange(steps + 1)
    v = np.maximum(cp * (S * u ** j * d ** (steps - j) - K), 0.0)       # payoffs at expiry
    for i in range(steps - 1, -1, -1):
        j = np.arange(i + 1)
        v = disc * (pu * v[1:] + (1 - pu) * v[:-1])
        if american:
            v = np.maximum(v, cp * (S * u ** j * d ** (i - j) - K))
    return float(v[0])

cases = [(100.0, 100.0, 1.0, 0.05, 0.0, 0.2, -1, 500, True), (100.0, 100.0, 1.0, 0.05, 0.0, 0.2, -1, 500, False),
         (100.0, 110.0, 0.5, 0.05, 0.02, 0.3, 1, 300, True)]
mine = [p.attempt(crr_price, *c) for c in cases]
mine = p.check("crr_price", mine, [p.crr_price(*c) for c in cases])
print(f"American put {mine[0]:.4f}, European put {mine[1]:.4f} (BSM {p.bsm_price(100, 100, 1, 0.05, 0, 0.2, -1):.4f}) → "
      f"early-exercise premium {mine[0] - mine[1]:.4f}")

In [ ]:
steps = np.arange(10, 301, 1)
err = [p.crr_price(100, 100, 1, 0.05, 0, 0.2, 1, s, american=False) - p.bsm_price(100, 100, 1, 0.05, 0, 0.2, 1) for s in steps]
Ks = np.linspace(70, 130, 25)
prem = [p.crr_price(100, k, 1, 0.05, 0, 0.2, -1, 300) - p.bsm_price(100, k, 1, 0.05, 0, 0.2, -1) for k in Ks]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(steps, err, lw=1); axes[0].axhline(0, color="black", lw=0.6)
axes[0].set(xlabel="steps", ylabel="tree − BSM", title="European call: converges, oscillating odd/even")
axes[1].plot(Ks, prem); axes[1].set(xlabel="strike (spot 100)", ylabel="American − European", title="Early-exercise premium of a put")
plt.tight_layout(); plt.show()

The premium grows with moneyness: a deep in-the-money put is worth exercising early to earn interest on the strike. That is why BSM implied vols of American puts come out too high (common mistake #5).

## 2. Monte Carlo and antithetic variates

Simulate terminal prices `S·exp((r − q − σ²/2)T + σ√T·z)`, average the discounted payoffs, report the standard error. **Antithetic variates:** for every `z` also use `−z`. Average each pair **first**: the pairs, not the individual paths, are the independent samples. Return `(price, standard error)`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def mc_antithetic(S, K, T, r, q, sigma, cp, n_paths=100_000, seed=0):
    rng = np.random.default_rng(seed)
    z = rng.standard_normal(n_paths // 2)
    drift, vol = (r - q - 0.5 * sigma ** 2) * T, sigma * np.sqrt(T)
    pay_up = np.maximum(cp * (S * np.exp(drift + vol * z) - K), 0)
    pay_dn = np.maximum(cp * (S * np.exp(drift - vol * z) - K), 0)
    x = np.exp(-r * T) * 0.5 * (pay_up + pay_dn)  # one discounted sample per pair
    return float(x.mean()), float(x.std(ddof=1) / np.sqrt(x.size))

mine = p.attempt(mc_antithetic, 100.0, 100.0, 1.0, 0.05, 0.0, 0.2, 1)
mine = p.check("mc_antithetic", mine, p.mc_european(100.0, 100.0, 1.0, 0.05, 0.0, 0.2, 1))
plain = p.mc_european(100.0, 100.0, 1.0, 0.05, 0.0, 0.2, 1, antithetic=False)
print(f"BSM {p.bsm_price(100, 100, 1, 0.05, 0, 0.2, 1):.4f}")
print(f"plain MC       {plain[0]:.4f} ± {plain[1]:.4f}")
print(f"antithetic MC  {mine[0]:.4f} ± {mine[1]:.4f}   (same 100,000 payoff evaluations, half the random draws)")

## 3. Greeks from simulation: common random numbers

A Monte Carlo delta by bump-and-revalue subtracts two noisy prices. With **different** random numbers for `S + h` and `S − h`, the noise of each price (a standard error of about 0.1 with 20,000 paths) is divided by `2h = 1` and swamps the answer. With the **same** random numbers the noise cancels.

In [ ]:
exact = p.greeks(100, 100, 1, 0.05, 0, 0.2, 1)["delta"]
same = [p.mc_delta(100, 100, 1, 0.05, 0, 0.2, 1, common=True, seed=s) for s in range(40)]
diff = [p.mc_delta(100, 100, 1, 0.05, 0, 0.2, 1, common=False, seed=s) for s in range(40)]
fig, ax = plt.subplots()
ax.hist(diff, bins=20, alpha=0.7, label=f"independent draws (sd {np.std(diff):.3f})")
ax.hist(same, bins=20, alpha=0.7, label=f"common random numbers (sd {np.std(same):.4f})")
ax.axvline(exact, color="black", lw=1, label=f"exact {exact:.4f}")
ax.set(xlabel="MC delta, 40 runs", title="Same work, very different Greeks"); ax.legend(); plt.show()

## Wrap-up

* Trees for early exercise; check them against BSM on European options.
* Monte Carlo: always report the standard error; antithetics are nearly free.
* Bumped MC Greeks need common random numbers (or pathwise / automatic differentiation).
* Graded version: `labs/part06/week22_greeks_numerics` (also Crank–Nicolson with Rannacher start-up and second-order convergence).